In [45]:
# ─── Step 0: 环境准备 & GPU 检测 ─────────────────────────────────────────────

# 安装依赖
!pip install -q sentence-transformers transformers tqdm scikit-learn rank_bm25

import torch, os
from google.colab import drive

# 挂载 Google Drive
drive.mount('/content/drive')

# 确认 GPU 可用
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
!nvidia-smi

# 定义路径
DRIVE_DIR = '/content/drive/MyDrive/NLP A3'
DATA_DIR  = os.path.join(DRIVE_DIR, 'data')
CLEAN_DIR = os.path.join(DRIVE_DIR, 'clean')
os.makedirs(CLEAN_DIR, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Tue May  6 13:46:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             34W /   70W |    6088MiB /  15360MiB |      0%      De

In [46]:
# Step 1: Data Cleaning & Saving (no NLTK)

import os, json, re, unicodedata
from tqdm import tqdm

DATA_DIR  = '/content/drive/MyDrive/NLP A3/data/'
CLEAN_DIR = '/content/drive/MyDrive/NLP A3/clean/'
os.makedirs(CLEAN_DIR, exist_ok=True)

def clean_text(text):
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^0-9A-Za-z\u4e00-\u9fa5\s\.,;:\-\'"]+', ' ', text)
    return text.lower().strip()

# Load raw data
with open(os.path.join(DATA_DIR,'train-claims.json'), 'r', encoding='utf-8') as f:
    train_raw = json.load(f)
with open(os.path.join(DATA_DIR,'dev-claims.json'), 'r', encoding='utf-8') as f:
    dev_raw = json.load(f)
with open(os.path.join(DATA_DIR,'evidence.json'), 'r', encoding='utf-8') as f:
    evidence_raw = json.load(f)

# Clean & save train-claims
train_claims_clean = {cid: clean_text(item['claim_text']) for cid,item in tqdm(train_raw.items(), desc='Clean train')}
with open(os.path.join(CLEAN_DIR,'train-claims-clean.json'),'w',encoding='utf-8') as f:
    json.dump(train_claims_clean, f, ensure_ascii=False, indent=2)

# Clean & save evidence
evi_clean = {eid: clean_text(txt) for eid,txt in tqdm(evidence_raw.items(), desc='Clean evidence')}
with open(os.path.join(CLEAN_DIR,'evidence-clean.json'),'w',encoding='utf-8') as f:
    json.dump(evi_clean, f, ensure_ascii=False, indent=2)

# Load dev claims raw text
dev_claims_text = {
    cid: clean_text(item['claim_text'])
    for cid, item in tqdm(dev_raw.items(), desc='Clean dev')
}

print("✅ Data cleaning done.")

Clean dev: 100%|██████████| 154/154 [00:00<00:00, 44635.67it/s]

✅ Data cleaning done.


In [53]:
# ─── Step 2: 检索 — 全 GPU Dense Retrieval + CrossEncoder 重排（不清洗 Dev） ─────────

import os, json
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

# 参数：初筛数量 & 重排 topN
INITIAL_K = 50
TOP_N     = 5

# 0) 设设备并初始化 SBERT 到 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device for retrieval:", device)
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# 1) 加载清洗后的 evidence 和 train-claims
CLEAN_DIR = '/content/drive/MyDrive/NLP A3/clean/'
with open(os.path.join(CLEAN_DIR, 'evidence-clean.json'), 'r', encoding='utf-8') as f:
    evi_clean = json.load(f)
with open(os.path.join(CLEAN_DIR, 'train-claims-clean.json'), 'r', encoding='utf-8') as f:
    train_claims_clean = json.load(f)

# 2) 加载原始 dev-claims 文本（无需清洗）
#    dev_claims_text 上一步 Step1 已经定义
# from Step1: dev_claims_text = {cid: raw_claim_text, ...}

# 3) 预计算 evidence 嵌入（一次性，GPU 上）
evi_ids   = list(evi_clean.keys())
evi_texts = [evi_clean[e] for e in evi_ids]
evi_embs  = sbert.encode(evi_texts, batch_size=64, convert_to_tensor=True, show_progress_bar=True)

# 4) Dense Retrieval 初筛 top INITIAL_K（GPU）
initial_train = {}
for cid, txt in tqdm(train_claims_clean.items(), desc='DenseRetr train'):
    q_emb = sbert.encode(txt, convert_to_tensor=True)
    scores = torch.nn.functional.cosine_similarity(q_emb, evi_embs)
    topk   = torch.topk(scores, k=INITIAL_K).indices
    initial_train[cid] = [evi_ids[i] for i in topk.cpu().tolist()]

initial_dev = {}
for cid, txt in tqdm(dev_claims_text.items(), desc='DenseRetr dev (raw)'):
    q_emb = sbert.encode(txt, convert_to_tensor=True)
    scores = torch.nn.functional.cosine_similarity(q_emb, evi_embs)
    topk   = torch.topk(scores, k=INITIAL_K).indices
    initial_dev[cid] = [evi_ids[i] for i in topk.cpu().tolist()]

# 5) CrossEncoder 重排（GPU + batching）
reranker = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    device=0 if device=="cuda" else -1
)

def cross_rerank(claims_map, initial_map, top_n=TOP_N, batch_size=64):
    reranked = {}
    for cid, docs in tqdm(initial_map.items(), desc='CrossEncoder rerank'):
        pairs  = [[claims_map[cid], evi_clean[d]] for d in docs]
        scores = reranker.predict(pairs, batch_size=batch_size)
        topk   = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)[:top_n]
        reranked[cid] = [d for d,_ in topk]
    return reranked

reranked_train = cross_rerank(train_claims_clean, initial_train)
reranked_dev   = cross_rerank(dev_claims_text,   initial_dev)

# （可选）保存重排结果
with open(os.path.join(CLEAN_DIR,'reranked_train.json'),'w',encoding='utf-8') as f:
    json.dump(reranked_train, f, ensure_ascii=False, indent=2)
with open(os.path.join(CLEAN_DIR,'reranked_dev.json'),'w',encoding='utf-8') as f:
    json.dump(reranked_dev,   f, ensure_ascii=False, indent=2)

print(f"✅ Step 2 完成：DenseRetr 初筛 top {INITIAL_K} + CrossEncoder 重排 top {TOP_N}。")


Using device for retrieval: cuda


Batches:   0%|          | 0/18888 [00:00<?, ?it/s]

CrossEncoder rerank: 100%|██████████| 154/154 [00:08<00:00, 17.71it/s]

✅ Step 2 完成：DenseRetr 初筛 top 50 + CrossEncoder 重排 top 5。


In [54]:
# ─── Step 3: 分类 — BERT Fine-tuning ─────────────────────────────────────────────

import torch
from torch.utils.data import Dataset
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments

# 3.1 准备金标
train_gold = {cid:item['claim_label'] for cid,item in train_raw.items()}
dev_gold   = {cid:item['claim_label'] for cid,item in dev_raw.items()}

# 3.2 Dataset
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
label2id  = {'SUPPORTS':0,'REFUTES':1,'NOT_ENOUGH_INFO':2,'DISPUTED':3}

class FactCheckDataset(Dataset):
    def __init__(self, claims_map, reranked_map, gold_map, tokenizer, max_len=256):
        self.cids      = list(claims_map.keys())
        self.claims    = claims_map
        self.reranked  = reranked_map
        self.gold      = gold_map
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, idx):
        cid   = self.cids[idx]
        text  = self.claims[cid] + ' [SEP] ' + ' [SEP] '.join(evi_clean[d] for d in self.reranked[cid])
        enc   = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item        = {k:v.squeeze(0) for k,v in enc.items()}
        item['labels'] = torch.tensor(label2id[self.gold[cid]])
        return item

train_ds = FactCheckDataset(train_claims_clean, reranked_train, train_gold, tokenizer)
dev_ds   = FactCheckDataset(dev_claims_text,   reranked_dev,   dev_gold,   tokenizer)

from transformers import TrainingArguments

# ─── Step 3.3 训练参数（关闭 W&B）────────────
training_args = TrainingArguments(
    output_dir                  = os.path.join(DRIVE_DIR,'bert-finetune'),
    num_train_epochs            = 4,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    learning_rate               = 3e-5,
    # 关键：不向任何远端报告（wandb、comet 等）
    report_to                   = [],
    # 如果你想看日志可以改成 ["tensorboard"] 并设置 logging_dir
    # report_to = ["tensorboard"],
    # logging_dir = "./logs",
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_ds,
    eval_dataset  = dev_ds,
    tokenizer     = tokenizer,
)
trainer.train()


<ipython-input-54-5d84832bd5a7>:60: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.609200


TrainOutput(global_step=616, training_loss=0.5480601694676783, metrics={'train_runtime': 322.4204, 'train_samples_per_second': 15.235, 'train_steps_per_second': 1.911, 'total_flos': 646212355915776.0, 'train_loss': 0.5480601694676783, 'epoch': 4.0})

In [55]:
import json
from tqdm import tqdm

predictions = {}
id2label    = {v:k for k,v in label2id.items()}
for cid, txt in tqdm(dev_claims_text.items(), desc='Predict dev'):
    # txt 已经是 clean_text 后的内容
    input_text = txt + ' [SEP] ' + ' [SEP] '.join(evi_clean[d] for d in reranked_dev[cid])
    enc = tokenizer(input_text, truncation=True, padding='max_length',
                    max_length=256, return_tensors='pt').to(device)
    pred = model(**enc).logits.argmax(dim=-1).item()
    predictions[cid] = {'claim_label': id2label[pred], 'evidences': reranked_dev[cid]}

out_path = "/content/drive/MyDrive/NLP A3/dev-claims-predictions_cleanDev.json"
with open(out_path, 'w') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

Predict dev: 100%|██████████| 154/154 [00:03<00:00, 50.74it/s]


In [56]:
# ─── Step 4b: 调用 eval.py 计算指标 ─────────────────────────────────────────────────
!python "/content/drive/MyDrive/NLP A3/eval.py" \
  --predictions "/content/drive/MyDrive/NLP A3/dev-claims-predictions_cleanDev.json"\
  --groundtruth "/content/drive/MyDrive/NLP A3/data/dev-claims.json"

Evidence Retrieval F-score (F)    = 0.19571222428365284
Claim Classification Accuracy (A) = 0.4025974025974026
Harmonic Mean of F and A          = 0.26338614527699417
